In [1]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate


class EmailClassification(BaseModel):
    category : Literal["Billing","Technical"]
    urgency: str

In [2]:
# EmailClassification(category='Technical',urgency='Low')

In [13]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

class EmailClassification(BaseModel):
    category : Literal["Billing","Technical"]
    urgency : Literal["Low","Medium","High"]
    confidence: int = Field(ge=1,le=10)


parser = PydanticOutputParser(pydantic_object=EmailClassification)

llm = ChatOpenAI(model_name="gpt-4o-mini")

In [14]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"category": {"enum": ["Billing", "Technical"], "title": "Category", "type": "string"}, "urgency": {"enum": ["Low", "Medium", "High"], "title": "Urgency", "type": "string"}, "confidence": {"maximum": 10, "minimum": 1, "title": "Confidence", "type": "integer"}}, "required": ["category", "urgency", "confidence"]}
```


In [16]:
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert support email classifier.

    Classification Rules:
    - Login broken after payment → Billing (NOT Technical)
    - App crashes → Technical
    - Pricing complaints + evaluating alternatives → Churn Risk
    - Feature requests → Feature Request

    Think step by step before classifying.
     {format_instructions}
    """),

    ("human", "Subject: {subject}\nBody: {body}")
]).partial(format_instructions=parser.get_format_instructions())

In [18]:
chain = cot_prompt | llm | parser

In [21]:
response = chain.invoke({"subject":"Login broken after payment",
              "body":"I paid for the app but now I can't log in. Please help!"})

In [24]:
response.urgency

'High'

In [25]:
response

EmailClassification(category='Billing', urgency='High', confidence=10)